In [1]:
from neo4j import GraphDatabase
import pandas as pd
import time
import os


In [2]:
driver = GraphDatabase.driver("neo4j://127.0.0.1:7687", auth=("neo4j", "sakuni200211"))

In [3]:
BASE_PATH = "D:/Downloads/Research Project/Data/processed/kg/"

In [4]:
FILES = ["triples_part_2.csv", "triples_part_3.csv", "triples_part_4.csv"]  # Remaining 3
BATCH_SIZE = 10000  # Small for low power

In [5]:
def bulk_load(tx, has_name_batch, has_brand_batch, contains_ing_batch):
    if not has_name_batch.empty:
        tx.run("UNWIND $data AS row MERGE (f:Food {id: row.sub}) ON CREATE SET f.name = row.obj", data=[{'sub': r['Subject'], 'obj': r['Object']} for _, r in has_name_batch.iterrows()])
    if not has_brand_batch.empty:
        tx.run("UNWIND $data AS row MERGE (f:Food {id: row.sub}) MERGE (b:Brand {name: row.obj}) MERGE (f)-[:HAS_BRAND]->(b)", data=[{'sub': r['Subject'], 'obj': r['Object']} for _, r in has_brand_batch.iterrows()])
    if not contains_ing_batch.empty:
        tx.run("UNWIND $data AS row MERGE (f:Food {id: row.sub}) MERGE (i:Ingredient {name: row.obj}) MERGE (f)-[:CONTAINS_INGREDIENT]->(i)", data=[{'sub': r['Subject'], 'obj': r['Object']} for _, r in contains_ing_batch.iterrows()])

start_time = time.time()
total_loaded = 0

In [ ]:
for file_name in FILES:
    csv_path = os.path.join(BASE_PATH, file_name)
    if not os.path.exists(csv_path):
        print(f"Skipping {file_name} (not found)")
        continue
    print(f"\n--- Loading {file_name} ({os.path.getsize(csv_path)/1e6:.1f} MB) ---")
    df = pd.read_csv(csv_path)
    df = df.sample(frac=0.1).reset_index(drop=True) 
    print(f"Using {len(df)} triples (sampled)")
    df_by_pred = {
        'has_name': df[df['Predicate'] == 'has_name'],
        'has_brand': df[df['Predicate'] == 'has_brand'],
        'contains_ingredient': df[df['Predicate'] == 'contains_ingredient']
    }


--- Loading triples_part_2.csv (251.4 MB) ---
Using 500000 triples (sampled)

--- Loading triples_part_3.csv (251.5 MB) ---
Using 500000 triples (sampled)

--- Loading triples_part_4.csv (256.1 MB) ---
Using 500000 triples (sampled)


In [7]:
file_start = time.time()

In [8]:
file_start = time.time()
    with driver.session() as session:
        for pred, df_pred in df_by_pred.items():
            print(f"  Loading {pred} ({len(df_pred)} rows)...")
            for i in range(0, len(df_pred), BATCH_SIZE):
                batch = df_pred.iloc[i:i + BATCH_SIZE]
                batch_dicts = {
                    'has_name_batch': pd.DataFrame() if pred != 'has_name' else batch,
                    'has_brand_batch': pd.DataFrame() if pred != 'has_brand' else batch,
                    'contains_ing_batch': pd.DataFrame() if pred != 'contains_ingredient' else batch
                }
                session.execute_write(bulk_load, **batch_dicts)
                if (i // BATCH_SIZE + 1) % 5 == 0:
                    elapsed = time.time() - file_start
                    print(f"    Batch {i // BATCH_SIZE + 1} ({i/len(df_pred)*100:.1f}%) - {elapsed/60:.1f} min for file")
            print(f"  {pred} done!")
    file_time = time.time() - file_start
    total_loaded += len(df)
    print(f"{file_name} loaded in {file_time/60:.1f} min")

IndentationError: unexpected indent (2074026405.py, line 2)

In [ ]:
for file_name in FILES:
    csv_path = os.path.join(BASE_PATH, file_name)
    if not os.path.exists(csv_path):
        print(f"Skipping {file_name} (not found)")
        continue
    print(f"\n--- Loading {file_name} ({os.path.getsize(csv_path)/1e6:.1f} MB) ---")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['Object']).reset_index(drop=True)
    df = df.sample(frac=0.1).reset_index(drop=True)  # Subsample
    print(f"Using {len(df)} triples (sampled)")
    
    df_by_pred = {
        'has_name': df[df['Predicate'] == 'has_name'],
        'has_brand': df[df['Predicate'] == 'has_brand'],
        'contains_ingredient': df[df['Predicate'] == 'contains_ingredient']
    }
    
    file_start = time.time()
    with driver.session() as session:
        for pred, df_pred in df_by_pred.items():
            print(f"  Loading {pred} ({len(df_pred)} rows)...")
            for i in range(0, len(df_pred), BATCH_SIZE):
                batch = df_pred.iloc[i:i + BATCH_SIZE]
                batch_dicts = {
                    'has_name_batch': pd.DataFrame() if pred != 'has_name' else batch,
                    'has_brand_batch': pd.DataFrame() if pred != 'has_brand' else batch,
                    'contains_ing_batch': pd.DataFrame() if pred != 'contains_ingredient' else batch
                }
                session.execute_write(bulk_load, **batch_dicts)
                if (i // BATCH_SIZE + 1) % 5 == 0:
                    elapsed = time.time() - file_start
                    print(f"    Batch {i // BATCH_SIZE + 1} ({i/len(df_pred)*100:.1f}%) - {elapsed/60:.1f} min for file")
            print(f"  {pred} done!")
    file_time = time.time() - file_start
    total_loaded += len(df)
    print(f"{file_name} loaded in {file_time/60:.1f} min")

total_time = time.time() - start_time
print(f"\nRemaining 3 files loaded in {total_time/60:.1f} min! Total triples from 3 files: {total_loaded}")
print("Verify: MATCH (f:Food) RETURN count(f)  # ~855K subsample (285K x 3)")

driver.close()


--- Loading triples_part_2.csv (251.4 MB) ---
Using 500000 triples (sampled)
  Loading has_name (48522 rows)...
    Batch 5 (82.4%) - 0.1 min for file
  has_name done!
  Loading has_brand (48448 rows)...
    Batch 5 (82.6%) - 0.3 min for file
  has_brand done!
  Loading contains_ingredient (403030 rows)...
    Batch 5 (9.9%) - 0.4 min for file
    Batch 10 (22.3%) - 0.6 min for file
    Batch 15 (34.7%) - 0.7 min for file
    Batch 20 (47.1%) - 0.8 min for file
    Batch 25 (59.5%) - 0.9 min for file
    Batch 30 (72.0%) - 1.0 min for file
    Batch 35 (84.4%) - 1.1 min for file
    Batch 40 (96.8%) - 1.3 min for file
  contains_ingredient done!
triples_part_2.csv loaded in 1.3 min

--- Loading triples_part_3.csv (251.5 MB) ---
Using 500000 triples (sampled)
  Loading has_name (48732 rows)...
    Batch 5 (82.1%) - 0.1 min for file
  has_name done!
  Loading has_brand (48024 rows)...
    Batch 5 (83.3%) - 0.2 min for file
  has_brand done!
  Loading contains_ingredient (403244 rows)...